# Notebook 06a — Evasion Attack Setup

Defines the attack configuration for NB06b (sensor-level PGD impersonation attack).

## Attack design

**Threat model**: member A walks with their own gait; the raw IMU sensor signal
is perturbed so that the authentication model accepts them as member B
(insider impersonation — one enrolled member's gait is modified to match another's).

**Surface**: raw IMU sensor signal only (6 channels × 128 timesteps).
The CNN representation layer is not attacked — perturbations must be injected
at the physical signal level before the CNN processes them.

**Perturbation norm: why L2, not L∞**:
L∞ bounds the maximum deviation at any single timestep, which concentrates
the perturbation budget into a small number of large spikes. For an IMU signal,
a spike corresponds to a brief impact event (footstrike, phone bump) and is easy
to detect or filter out. L2 bounds the total perturbation *energy*, producing a
smooth, low-amplitude modification spread across the entire window, physically
equivalent to a subtle continuous change in walking style (speed, posture, stride),
which is indistinguishable from the sensor's own thermal noise floor and natural
intra-session gait variation. Forestier et al. (2022) show empirically that L2/smooth
perturbations are the only adversarial perturbations that remain imperceptible on
time-series signals, while L∞ attacks always produce detectable artefacts.

## References

- **PGD attack**: Madry et al., *Towards Deep Learning Models Resistant to
  Adversarial Attacks*, ICLR 2018.
- **L2 norm for time-series**: Forestier et al., *Smooth Perturbations for Time
  Series Adversarial Attacks*, PAKDD 2022.
- **L2 attack (C&W)**: Carlini & Wagner, *Towards Evaluating the Robustness of
  Neural Networks*, IEEE S&P 2017.
- **Closest domain**: Huang et al., *Evaluating Deep Learning Models and
  Adversarial Attacks on Accelerometer-Based Gesture Authentication*, 2021.

## What this notebook produces

1. ε_target: median intra-subject L2 distance on same-person pairs: the
   perturbation threshold below which an attack is smaller than the typical spread
   of a subject's own gait windows (Section 2).
2. ε grid: exponential grid concentrated near ε_target used by the binary
   search in NB06b to find the minimum ε that fools the model per pair (Section 3).
3. Impostor combo inventory: all (A,B) subject pairs in D5 with their
   sensor-space L2 difficulty and baseline model confidence (Sections 4–5).

Outputs saved to `logs/06a_attack_setup.npz`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json, logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

import torch

from src.data.auth_dataset  import load_auth_dataset, normalize_auth
from src.models.gait_cnn   import GaitCNN
from src.models.auth_model import AuthModel
from src.utils.latex_writer import write_latex_metrics

# ── Dataset configuration ─────────────────────────────────────────────────────
# Change these lines to run on a different dataset.
DATASET_NAME = 'Dataset #5'
DATA_ROOT    = Path('../data')
ATTR_PATH    = Path('../logs/d5_attribution.npz')   # built by NB01 for this dataset

# Derived paths (dataset-agnostic)
DATASET_PATH = DATA_ROOT / DATASET_NAME
NORM_PATH    = Path('../logs/auth_norm_stats.npz')
LOG_DIR      = Path('../logs')
CKPT_DIR     = Path('../checkpoints')
RESULT_DIR   = Path('../results')

# Signal shape — update if the new dataset uses different dimensions
N_CHANNELS  = 6     # IMU channels (acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z)
N_TIMESTEPS = 128   # window length in samples

BATCH  = 512
DEVICE = torch.device('cpu')

# ── Activity configuration — controls how ε_target is computed ────────────────
# Dataset #5: no activity labels → set to None → global median fallback (Section 2).
# UCI-HAR / WISDM: set ACTIVITY_LABEL to the walking label string and
#   ACTIVITY_LABELS_TRAIN to the (N_train,) array of activity labels aligned
#   with the training pairs.  The notebook will filter same-person pairs to that
#   activity and take the median — no global contamination from other activities.
ACTIVITY_LABEL        = None   # e.g. 'WALKING' for UCI-HAR, 'Walking' for WISDM
ACTIVITY_LABELS_TRAIN = None   # (N_train,) array of activity labels per pair; None = not available

# ── Grid hyperparameters ──────────────────────────────────────────────────────
N_GRID     = 12    # grid points from EPS_FLOOR to eps_target
N_BISECT   = 6     # binary search refinement steps per pair
FLOOR_FRAC = 0.05  # EPS_FLOOR = eps_target * FLOOR_FRAC

log = logging.getLogger('nb06a')
log.setLevel(logging.DEBUG)
log.handlers.clear()
fh = logging.FileHandler(LOG_DIR / f'06a_{DATASET_NAME.replace(" ","_")}_attack_setup.log', mode='w')
fh.setFormatter(logging.Formatter('%(asctime)s  %(message)s', datefmt='%H:%M:%S'))
sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter('%(message)s'))
log.addHandler(fh); log.addHandler(sh)
log.info(f'=== Notebook 06a — Evasion Attack Setup  [{DATASET_NAME}] ===')
log.info(f'Activity mode: {"filter → " + str(ACTIVITY_LABEL) if ACTIVITY_LABEL else "global median (no labels)"}')

## Section 1 — Load Data and Model

We load both splits for different purposes:
- **Train split** (members, seen by model): used only to compute ε_target from
  same-person pairs, a property of the signal distribution, not of the model.
- **Test split** (non-members, never seen by model): used for the impostor combo
  analysis and the attack evaluation. The model's decision boundary on these pairs
  reflects true generalisation, not memorisation of training subjects.

In [2]:
norm_stats        = np.load(NORM_PATH)
norm_mean, norm_std = norm_stats['mean'], norm_stats['std']

# Train split — members (used for ε_target only)
X1_tr, X2_tr, y_tr = load_auth_dataset(str(DATASET_PATH), 'train')
X1_tr_n, X2_tr_n, _ = normalize_auth(X1_tr, X2_tr, (norm_mean, norm_std))

# Test split — non-members (used for attack evaluation)
X1_te, X2_te, y_te = load_auth_dataset(str(DATASET_PATH), 'test')
X1_te_n, X2_te_n, _ = normalize_auth(X1_te, X2_te, (norm_mean, norm_std))

log.info(f'[{DATASET_NAME}] train: {len(y_tr):,} pairs  '
         f'same={( y_tr==0).sum():,}  diff={(y_tr==1).sum():,}')
log.info(f'[{DATASET_NAME}] test : {len(y_te):,} pairs  '
         f'same={( y_te==0).sum():,}  diff={(y_te==1).sum():,}')

# Attribution arrays (built by NB01 for this dataset)
_attr        = np.load(ATTR_PATH)
subj_w1_tr   = _attr['subj_win1_train']
subj_w2_tr   = _attr['subj_win2_train']
subj_w1_te   = _attr['subj_win1_test']
subj_w2_te   = _attr['subj_win2_test']

# Load frozen CNN encoder and authenticator
cnn = GaitCNN(n_classes=98)
cnn.load_state_dict(torch.load(CKPT_DIR / 'cnn_encoder.pt', map_location='cpu'))
cnn.eval()

auth_model = AuthModel(cnn).to(DEVICE)
auth_model.load_state_dict(torch.load(CKPT_DIR / 'auth_model.pt', map_location='cpu'))
auth_model.eval()

print(f'Train: {len(y_tr):,} pairs  (same={( y_tr==0).sum():,}, diff={(y_tr==1).sum():,})')
print(f'Test : {len(y_te):,} pairs  (same={( y_te==0).sum():,}, diff={(y_te==1).sum():,})')

[Dataset #5] train: 66,542 pairs  same=33,271  diff=33,271
[Dataset #5] test : 7,600 pairs  same=3,800  diff=3,800


Train: 66,542 pairs  (same=33,271, diff=33,271)
Test : 7,600 pairs  (same=3,800, diff=3,800)


## Section 2 — ε_target: Intra-subject L2 as the Perturbation Reference

ε_target is the median L2 distance between two windows from the same person.
It serves as the reference scale for the attack: a perturbation below ε_target
is smaller than the typical spread of a subject's own gait windows and therefore
has a plausible claim to imperceptibility.

### Why the global median suffices for Dataset #5

We investigated whether a Gaussian Mixture Model (k=2) could separate a
*within-session* component (low L2) from a *cross-session* component (high L2)
in the intra-subject distribution, the hypothesis being that cross-session pairs
inflate the median and make ε_target too loose.

The result was essentially unimodal: the lower GMM component captured 94.2 % of
the data with μ=25.0, nearly identical to the global median (24.5). No meaningful
second mode exists, confirming that the intra-subject L2 distribution in Dataset #5
is well described by a single Gaussian. The global median is therefore the correct
and sufficient estimator for ε_target.

> **Why is intra-subject L2 as high as ~25?**  
> Same-person pairs are formed by sampling two *non-consecutive* windows from the
> same long recording.  Gait is periodic (~1–2 cycles per 128-sample window), so
> a phase offset between windows produces a large raw L2 even within the same bout:
> a window starting at heel-strike looks very different from one starting at
> mid-swing, even for the same person walking steadily.  This **phase misalignment
> artefact** is the dominant source of intra-subject variability in raw sensor
> space, not cross-session variation.  
> The model is robust to this because its CNN+LSTM learns phase-invariant features;
> the attack operates in raw sensor space where phase matters.

### Two-path computation (controlled by config)

**Path A — activity labels available** (`ACTIVITY_LABEL` is set):  
Filter same-person pairs to the target activity and take the median.
Eliminates cross-activity inflation (running, stairs, etc.).
Used for UCI-HAR (`'WALKING'`) and WISDM (`'Walking'`) [they will be later implemented].

**Path B — no labels** (`ACTIVITY_LABEL = None`, current dataset):  
Global median of all intra-subject L2 pairs.

> **Limitation**: ε_target alone is an imperfect quality bar — see Section 2b.

In [ ]:
same_mask_tr = y_tr == 0
x1_same      = X1_tr_n[same_mask_tr]
x2_same      = X2_tr_n[same_mask_tr]

l2_same = np.linalg.norm(
    (x1_same - x2_same).reshape(len(x1_same), -1), axis=1
)
eps_global_median = float(np.median(l2_same))

# ── Path A: activity labels available ─────────────────────────────────────────
if ACTIVITY_LABEL is not None:
    act_same    = np.asarray(ACTIVITY_LABELS_TRAIN)[same_mask_tr]
    l2_filtered = l2_same[act_same == ACTIVITY_LABEL]
    eps_target  = float(np.median(l2_filtered))
    eps_method  = f'activity_filter ({ACTIVITY_LABEL})'

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(l2_same,     bins=80, density=True, color='#bdc3c7', alpha=0.5,
            label='all same-person pairs')
    ax.hist(l2_filtered, bins=80, density=True, color='#3498db', alpha=0.7,
            label=f'filtered: {ACTIVITY_LABEL}')
    ax.axvline(eps_target,        color='#3498db', linewidth=2, linestyle='--',
               label=f'ε_target = {eps_target:.2f} (filtered median)')
    ax.axvline(eps_global_median, color='gray',    linewidth=1.5, linestyle=':',
               label=f'global median = {eps_global_median:.2f}')

# ── Path B: no labels — global median ────────────────────────────────────────
else:
    eps_target = eps_global_median
    eps_method = 'global_median'

    p10, p25, p50, p75, p90 = np.percentile(l2_same, [10, 25, 50, 75, 90])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(l2_same, bins=80, density=True, color='#bdc3c7', alpha=0.7,
            edgecolor='none', label='same-person pairs')
    ax.axvline(eps_target, color='#3498db', linewidth=2, linestyle='--',
               label=f'ε_target (median) = {eps_target:.2f}')
    ax.axvspan(p25, p75, alpha=0.12, color='#3498db', label=f'IQR [{p25:.1f}, {p75:.1f}]')

# ── Shared plot formatting ────────────────────────────────────────────────────
ax.set_xlabel(f'L2 distance — normalised sensor space ({N_CHANNELS}×{N_TIMESTEPS} dims)')
ax.set_ylabel('Density')
ax.set_title(f'Intra-subject L2 distribution — {eps_method}  [{DATASET_NAME} train]')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / f'06a_{DATASET_NAME.replace(" ","_")}_l2_distribution.png', dpi=150)
plt.show()

log.info(f'ε_target = {eps_target:.3f}  method={eps_method}')
log.info(f'Intra-subject L2: p10={np.percentile(l2_same,10):.3f}  '
         f'median={eps_target:.3f}  p90={np.percentile(l2_same,90):.3f}')
print(f'ε_target = {eps_target:.3f}  [{eps_method}]')

## Section 2b — Evaluating Attack Quality: Beyond ε_target

ε_target is used to build the grid and define the floor — it is not a rigorous
imperceptibility guarantee. The phase misalignment artefact means two windows from
the same person already differ by ~25 L2 units for geometric reasons (phase offset),
not because that is a physically large perturbation. Reporting ε_min / ε_target
alone would overstate imperceptibility relative to an inflated reference.

We therefore report **three complementary metrics** in NB06b:

### Metric 1 — ε_min / ε_target (grid scale ratio)

How much of the intra-subject variation budget does the attack consume?

$$r = \varepsilon_{\min} / \varepsilon_{\text{target}}$$

Useful for comparing across combos and datasets (dimensionless and self-normalised),
but r is loose when ε_target includes phase-misalignment artefacts.

### Metric 2 — Perturbation-to-Signal Ratio (PSR)

$$\text{PSR} = \frac{\|\delta\|_2}{\|x_1\|_2}$$

Ratio of perturbation energy to the original signal energy, averaged over pairs.
Independent of ε_target. A PSR of 2% means the attack adds energy equivalent to
2% of the raw IMU signal — a concrete physical quantity.

Already implemented in `src/attacks/pgd.py:psr()`.

### Metric 3 — Perturbation relative to inter-subject distance

$$r_{\text{inter}} = \varepsilon_{\min} / \text{median}_{(A,B)}\,\|x_1^A - x_2^B\|_2$$

The denominator is the typical distance between different people's gait windows
(the combo difficulty distribution, Section 4). If the attack requires a perturbation
that is only 7% of the between-person distance, it is small relative to the
discrimination gap the model is solving.

This metric is computed in NB06b using the per-combo L2 difficulty values saved here.

Together, PSR gives the physical scale; ε_min/ε_target gives the grid-relative scale;
r_inter gives the discrimination-relative scale.

## Section 3 — ε Grid Construction

The grid spans `[EPS_FLOOR, ε_target]` plus sparse sentinels above:

- **EPS_FLOOR = ε_target × 5%**: a data-driven floor, below 5% of ε_target the
  perturbation is smaller than 1/20th of intra-subject L2 variation. The grid
  provides no information below this level.
- **Dense below ε_target** (`N_GRID` points, geomspace): each step is ×1.3,
  providing the finest resolution in the sub-ε_target range.
- **Sparse sentinels above** (2×, 5×, 10× ε_target): classify pairs that only
  fail at large budgets as *resistant*.

The binary search in NB06b refines within the bracket `[ε_failed, ε_success]`
found by this grid.

In [ ]:
EPS_FLOOR      = eps_target * FLOOR_FRAC
grid_below     = np.geomspace(EPS_FLOOR, eps_target, N_GRID)
grid_sentinels = np.array([eps_target * 2, eps_target * 5, eps_target * 10])
EPS_GRID       = np.unique(np.concatenate([grid_below, grid_sentinels]))

step_ratio = (eps_target / EPS_FLOOR) ** (1 / (N_GRID - 1))

log.info(f'EPS_FLOOR = {EPS_FLOOR:.3f}  (ε_target × {FLOOR_FRAC:.0%})')
log.info(f'Grid step ratio ×{step_ratio:.2f}  ({N_GRID} dense + {len(grid_sentinels)} sentinels)')
log.info(f'ε grid: {np.round(EPS_GRID, 3).tolist()}')

fig, ax = plt.subplots(figsize=(9, 2.5))
ax.scatter(EPS_GRID[:N_GRID], np.zeros(N_GRID),
           color='#3498db', s=60, zorder=3, label=f'dense ({N_GRID} pts)')
ax.scatter(EPS_GRID[N_GRID:], np.zeros(len(EPS_GRID) - N_GRID),
           color='#e74c3c', s=80, marker='x', zorder=3, linewidths=2,
           label='sentinels')
ax.axvline(EPS_FLOOR,  color='gray',    linewidth=1, linestyle='--',
           label=f'floor={EPS_FLOOR:.2f} ({FLOOR_FRAC:.0%} of target)')
ax.axvline(eps_target, color='#e74c3c', linewidth=1.5, linestyle='--',
           label=f'ε_target={eps_target:.2f}')
ax.set_xlabel('ε (L2 budget)'); ax.set_yticks([])
ax.set_xscale('log')
ax.set_title('ε grid — geomspace from data-driven floor to ε_target, sparse sentinels above')
ax.legend(fontsize=8, loc='upper left'); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(RESULT_DIR / f'06a_{DATASET_NAME.replace(" ","_")}_eps_grid.png', dpi=150)
plt.show()

print(f'EPS_FLOOR = {EPS_FLOOR:.3f}  |  ε_target = {eps_target:.3f}  |  '
      f'step ×{step_ratio:.2f}  |  total grid points: {len(EPS_GRID)}')

## Section 4 — Impostor Combo Inventory and Difficulty (Test Split)

### What is an impostor combo?

An **impostor combo** is a unique ordered pair `(A, B)` of subject IDs
appearing in the impostor pairs (y=1) of the **test split**:

- `x1` → **subject A** — the probe window that will be perturbed by PGD
- `x2` → **subject B** — the reference stored in the authentication database (untouched)

We use the **test split** because its subjects were never seen by the model during
training. The model's decision boundary on these pairs reflects true generalisation,
not memorisation of training subjects.

Within a combo, D5 samples multiple (x1, x2) window pairs from the same (A, B)
subject pair. This gives several independent pairs per combo to average the
attack success rate over.

### Difficulty metric

```
difficulty(A,B) = mean over pairs of ‖x1_A − x2_B‖₂   (sensor space, same norm as ε)
```

Combos with low difficulty are structurally close in sensor space, a small
perturbation may suffice. Combos with high difficulty require a larger budget
or may be resistant altogether.

In [5]:
# Work on the test split — non-members, never seen by the model
diff_mask_te = y_te == 1
s1_arr_te    = subj_w1_te[diff_mask_te]
s2_arr_te    = subj_w2_te[diff_mask_te]
diff_indices_te = np.where(diff_mask_te)[0]

x1_diff_te = X1_te_n[diff_mask_te]
x2_diff_te = X2_te_n[diff_mask_te]

pair_l2_te = np.linalg.norm(
    (x1_diff_te - x2_diff_te).reshape(len(x1_diff_te), -1), axis=1
)

combo_to_local = defaultdict(list)
for loc, (a, b) in enumerate(zip(s1_arr_te.tolist(), s2_arr_te.tolist())):
    combo_to_local[(int(a), int(b))].append(loc)

combo_records = []
for (a, b), locs in combo_to_local.items():
    orig_idxs = [int(diff_indices_te[l]) for l in locs]
    combo_records.append({
        'subj_a':        a,
        'subj_b':        b,
        'n_pairs':       len(locs),
        'l2_dist':       float(pair_l2_te[locs].mean()),
        'pair_indices':  orig_idxs,
        'local_indices': locs,
    })

combo_records.sort(key=lambda r: r['l2_dist'])

l2_vals      = np.array([r['l2_dist']  for r in combo_records])
n_pairs_vals = np.array([r['n_pairs']  for r in combo_records])

log.info(f'Test impostor combos: {len(combo_records):,}')
log.info(f'Pairs per combo: min={n_pairs_vals.min()}  '
         f'median={np.median(n_pairs_vals):.0f}  max={n_pairs_vals.max()}')
log.info(f'L2 difficulty: min={l2_vals.min():.2f}  '
         f'median={np.median(l2_vals):.2f}  max={l2_vals.max():.2f}')

print(f'Test impostor combos: {len(combo_records):,}')
print(f'Total test impostor pairs: {diff_mask_te.sum():,}')
print(f'Pairs per combo: min={n_pairs_vals.min()}  '
      f'median={np.median(n_pairs_vals):.0f}  max={n_pairs_vals.max()}')
print(f'L2 difficulty: {l2_vals.min():.2f} → {l2_vals.max():.2f}  '
      f'(ratio {l2_vals.max()/l2_vals.min():.1f}×)')
print(f'ε_target={eps_target:.3f}  |  combos with difficulty < ε_target: '
      f'{(l2_vals < eps_target).sum()} ({(l2_vals < eps_target).mean()*100:.1f}%)')

Test impostor combos: 220
Pairs per combo: min=1  median=10  max=302
L2 difficulty: min=15.75  median=41.17  max=56.25


Test impostor combos: 220
Total test impostor pairs: 3,800
Pairs per combo: min=1  median=10  max=302
L2 difficulty: 15.75 → 56.25  (ratio 3.6×)
ε_target=24.536  |  combos with difficulty < ε_target: 2 (0.9%)


In [ ]:
q25, q50, q75 = np.percentile(l2_vals, [25, 50, 75])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(l2_vals, bins=60, color='#3498db', alpha=0.8, edgecolor='none')
for q, lbl, ls in [(q25, 'Q1', '--'), (q50, 'median', '-'), (q75, 'Q3', '--')]:
    ax.axvline(q, color='#e74c3c', linewidth=1.5, linestyle=ls, label=f'{lbl}={q:.1f}')
ax.axvline(eps_target, color='#2ecc71', linewidth=1.5, linestyle='-.',
           label=f'ε_target={eps_target:.2f}')
ax.set_xlabel('Mean sensor-space L2(x1_A, x2_B) per combo')
ax.set_ylabel('Combo count')
ax.set_title('Impostor combo difficulty distribution')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
unique_counts, cnt = np.unique(n_pairs_vals, return_counts=True)
ax.bar(unique_counts, cnt, color='#2ecc71', alpha=0.8, edgecolor='none')
ax.set_xlabel('Pairs per (A,B) combo')
ax.set_ylabel('Number of combos')
ax.set_title('Pairs per combo (expected: 7 throughout)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(RESULT_DIR / '06a_combo_analysis.png', dpi=150)
plt.show()

log.info(f'L2 quartiles: Q1={q25:.2f}  Q2={q50:.2f}  Q3={q75:.2f}')

## Section 5 — Baseline Model Scores on Impostor Pairs

Before any attack, the model outputs `softmax[:,1]` = P(different person) for each pair.
On impostor pairs (y=1) the model should output a high P(different), it correctly
recognises them as different people. The attack goal is to push this **below 0.5**
so the model accepts the pair as genuine.

1. Pairs where P(different) is already close to 0.5 sit near the decision boundary
   and need very little perturbation to cross it.
2. If the model is more uncertain on combos with small L2 distance, the difficulty
   metric is well-calibrated.

In [ ]:
from src.attacks.pgd import batch_psame   # softmax[:,1] = P(different person)

X1_diff_t = torch.from_numpy(x1_diff_te).float()
X2_diff_t = torch.from_numpy(x2_diff_te).float()

p_diff_baseline = batch_psame(auth_model, X1_diff_t, X2_diff_t, batch_size=BATCH)

print(f'Baseline P(different) on test impostor pairs:')
print(f'  mean={p_diff_baseline.mean():.4f}  std={p_diff_baseline.std():.4f}')
print(f'  already accepted (P<0.5): {(p_diff_baseline < 0.5).mean()*100:.2f}%')

# Per-combo mean baseline score
combo_baseline = []
for r in combo_records:
    locs = r['local_indices']
    combo_baseline.append(float(p_diff_baseline[locs].mean()))
combo_baseline = np.array(combo_baseline)

fig, ax = plt.subplots(figsize=(8, 4))
sc = ax.scatter(l2_vals, combo_baseline, c=combo_baseline, cmap='RdYlGn',
                vmin=0, vmax=1, s=10, alpha=0.6)
ax.axhline(0.5, color='black', linewidth=1.5, linestyle='--',
           label='decision boundary (P=0.5)')
ax.axvline(eps_target, color='gray', linewidth=1, linestyle=':',
           alpha=0.8, label=f'ε_target={eps_target:.2f}')
ax.set_xlabel('Mean L2(x1_A, x2_B) per combo — difficulty')
ax.set_ylabel('Mean P(different) — baseline')
ax.set_title(f'Baseline model confidence vs combo difficulty [{DATASET_NAME} test]\n'
             'green = near boundary (easier to attack)   red = confident rejection (harder)')
plt.colorbar(sc, ax=ax, label='P(different)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / f'06a_{DATASET_NAME.replace(" ","_")}_baseline_scores.png', dpi=150)
plt.show()

log.info(f'Baseline P(diff) on test impostor pairs: '
         f'mean={p_diff_baseline.mean():.4f}  '
         f'already_accepted={(p_diff_baseline<0.5).mean():.4f}')

## Section 6 — Save Setup for NB06b

In [ ]:
q25, q50, q75 = np.percentile(l2_vals, [25, 50, 75])

combo_subj_a          = np.array([r['subj_a']  for r in combo_records], dtype=np.int32)
combo_subj_b          = np.array([r['subj_b']  for r in combo_records], dtype=np.int32)
combo_n_pairs         = np.array([r['n_pairs'] for r in combo_records], dtype=np.int32)
combo_l2              = l2_vals.astype(np.float32)
combo_baseline_scores = combo_baseline.astype(np.float32)
diff_pair_indices_te  = np.where(diff_mask_te)[0].astype(np.int32)

out_path = LOG_DIR / f'06a_{DATASET_NAME.replace(" ","_")}_attack_setup.npz'
np.savez(
    out_path,
    dataset_name          = DATASET_NAME,
    eps_grid              = EPS_GRID.astype(np.float32),
    eps_target            = np.float32(eps_target),
    eps_floor             = np.float32(EPS_FLOOR),
    floor_frac            = np.float32(FLOOR_FRAC),
    n_bisect              = np.int32(N_BISECT),
    l2_same               = l2_same.astype(np.float32),
    combo_subj_a          = combo_subj_a,
    combo_subj_b          = combo_subj_b,
    combo_n_pairs         = combo_n_pairs,
    combo_l2              = combo_l2,
    combo_baseline_scores = combo_baseline_scores,
    diff_pair_indices_te  = diff_pair_indices_te,
    subj_w1_diff_te       = s1_arr_te.astype(np.int32),
    subj_w2_diff_te       = s2_arr_te.astype(np.int32),
)
print(f'Saved: {out_path}')

tag = DATASET_NAME.replace(' ', '_').replace('#', '')
metrics = {
    'datasetName':              DATASET_NAME,
    'epsTarget':                f'{eps_target:.3f}',
    'epsTargetMethod':          eps_method,
    'epsFloor':                 f'{EPS_FLOOR:.3f}',
    'floorFracPct':             f'{FLOOR_FRAC*100:.0f}',
    'l2SameMean':               f'{l2_same.mean():.3f}',
    'l2SameP10':                f'{np.percentile(l2_same,10):.3f}',
    'l2SameP90':                f'{np.percentile(l2_same,90):.3f}',
    'nImpostorCombos':          int(len(combo_records)),
    'nImpostorPairsTest':       int(diff_mask_te.sum()),
    'comboDiffL2Min':           f'{l2_vals.min():.2f}',
    'comboDiffL2Median':        f'{q50:.2f}',
    'comboDiffL2Max':           f'{l2_vals.max():.2f}',
    'comboDiffL2Q1':            f'{q25:.2f}',
    'comboDiffL2Q3':            f'{q75:.2f}',
    'epsGridN':                 int(len(EPS_GRID)),
    'nBisect':                  int(N_BISECT),
    'gridStepRatio':            f'{step_ratio:.2f}',
    'baselinePdiff':            f'{p_diff_baseline.mean():.4f}',
    'baselineAlreadyAccepted':  f'{(p_diff_baseline < 0.5).mean():.4f}',
}
write_latex_metrics(f'nb06a_{tag}', metrics, output_dir='../latex/generated', log=log)
print('LaTeX macros written.')
log.info('=== NB06a complete ===')